# Bad Apple → JSON para Google Sheets

Converte o vídeo do **Bad Apple!!** em uma lista de frames em preto e branco, no formato que o Apps Script do projeto usa para desenhar a animação nas células do Google Sheets.

**Formato de saída:** um array de frames; cada frame é um array de `HEIGHT` strings com `WIDTH` caracteres, onde `"1"` é pixel preto e `"0"` é pixel branco.

```json
[
  ["0000...0000", "0001...1000", ...],
  ...
]
```

> Feito para rodar no **Google Colab** (usa `google.colab.files` para upload/download).

## 1. Instalar dependências

In [1]:
# Dependências: OpenCV para ler/processar o vídeo e tqdm para a barra de progresso
!pip install opencv-python tqdm

## 2. Enviar o vídeo

Selecione o arquivo `.mp4` do Bad Apple.

In [2]:
# Abre o seletor de arquivos do Colab para enviar o vídeo (ex.: badApple.mp4)
from google.colab import files

uploaded = files.upload()

Saving badApple.mp4 to badApple.mp4


## 3. Configuração

Ajuste a resolução e o FPS do resultado. Quanto maior a resolução e o FPS, maior o JSON e mais pesada a animação na planilha.

| Arquivo em `data/` | WIDTH × HEIGHT | FPS |
|---|---|---|
| `video_40_30_10fps.json` | 40 × 30 | 10 |
| `video_80_60_10fps.json` | 80 × 60 | 10 |
| `video_80_60_15fps.json` | 80 × 60 | 15 |

In [ ]:
# Nome do vídeo enviado na etapa anterior
VIDEO_FILE = "badApple.mp4"

# Resolução da "tela" na planilha: 1 pixel = 1 célula
# (WIDTH colunas x HEIGHT linhas)
WIDTH = 80
HEIGHT = 60

# Frames por segundo desejados no JSON final.
# O vídeo original tem 30 fps; os frames excedentes são descartados.
TARGET_FPS = 15

# Limiar de binarização (0-255): tons de cinza abaixo disso viram preto
THRESHOLD = 128

## 4. Converter os frames

Para cada frame mantido: tons de cinza → redimensiona → binariza (preto/branco) → codifica cada linha como string de `0`/`1`.

In [4]:
import cv2
import json
from tqdm import tqdm

cap = cv2.VideoCapture(VIDEO_FILE)

original_fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Mantém 1 a cada `frame_skip` frames para chegar perto do TARGET_FPS
# (ex.: 30 fps -> 10 fps = pular de 3 em 3)
frame_skip = max(1, int(original_fps / TARGET_FPS))

print(f"FPS original: {original_fps}")
print(f"Frames originais: {total_frames}")
print(f"Pular a cada: {frame_skip}")

# Lista final: cada frame é uma lista de HEIGHT strings com WIDTH caracteres
frames = []

current_frame = 0

with tqdm(total=total_frames) as pbar:
    while True:
        ret, frame = cap.read()

        # Fim do vídeo
        if not ret:
            break

        if current_frame % frame_skip == 0:

            # 1. Converte para tons de cinza
            gray = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2GRAY
            )

            # 2. Reduz para a resolução da planilha
            resized = cv2.resize(
                gray,
                (WIDTH, HEIGHT)
            )

            # 3. Binariza: cada pixel vira 0 (preto) ou 255 (branco)
            _, bw = cv2.threshold(
                resized,
                THRESHOLD,
                255,
                cv2.THRESH_BINARY
            )

            # 4. Codifica cada linha como texto: "1" = preto, "0" = branco
            encoded = [
                "".join(
                    "1" if pixel == 0 else "0"
                    for pixel in row
                )
                for row in bw
            ]

            frames.append(encoded)

        current_frame += 1
        pbar.update(1)

cap.release()

print(f"\nFrames convertidos: {len(frames)}")

FPS original: 30.0
Frames originais: 6572
Pular a cada: 3


100%|██████████| 6572/6572 [00:07<00:00, 920.11it/s] 


Frames convertidos: 2191


## 5. Salvar e baixar o JSON

In [5]:
# Salva todos os frames num único JSON.
# Renomeie depois seguindo o padrão video_<W>_<H>_<FPS>fps.json e coloque em data/
with open("frames.json", "w") as f:
    json.dump(frames, f)

print("frames.json criado!")

frames.json criado!


In [6]:
# Baixa o JSON gerado para o computador
from google.colab import files

files.download("frames.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Pré-visualizar um frame

In [7]:
# Pré-visualiza um frame no terminal para conferir o resultado
# (troque o índice para ver outros momentos do vídeo)
frame = frames[500]

for row in frame:
    print(
        row.replace("1", "█")
           .replace("0", " ")
    )

████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
███████████████████                     
██████████████████████                  
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████████████████████████████
████████████████